In [0]:
from pyspark.sql.functions import col, current_timestamp, to_json, struct, lit, trim, upper, when, to_date, coalesce, try_to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

vendor_name = "nv5"
layer_name = "silver_conform"

print(f"--- Starting Silver Conformance for: {vendor_name} ---")

# 1. Read from Bronze table
df_bronze = spark.read.table("inlap.bronze.nv5")

# 2. Type casting, normalization, and canonical gold-join columns
df_cleaned = (
    df_bronze
    .withColumn("latitude", col("lat").cast("double"))
    .withColumn("longitude", col("lon").cast("double"))
    .withColumn("addr", trim(col("addr")))
    .withColumn("desc", trim(col("desc")))
    .withColumn("cty", upper(trim(col("cty"))))
    .withColumn("st", upper(trim(col("st"))))
    .withColumn(
        "event_date",
        coalesce(
            try_to_date(col("survey_dt").cast("string"), "yyyyMMdd"),
            try_to_date(col("survey_dt").cast("string"), "yyyy-MM-dd HH:mm:ss"),
            to_date(col("survey_dt"))
        )
    )
    .withColumn("event_timestamp", col("survey_dt").cast("timestamp"))
    .withColumn("_truncated_source_field", lit(True))
    .withColumn(
        "normalized_status",
        when(upper(trim(col("status"))).isin("ACTIVE", "ACT"), "Active")
        .when(upper(trim(col("status"))).isin("DECOMMISSIONED", "DECOMM", "INACTIVE"), "Decommissioned")
        .when(upper(trim(col("status"))).isin("UNDER MAINTENANCE", "MAINTENANCE", "MAINT"), "Under Maintenance")
        .when(upper(trim(col("status"))).isin("PLANNED", "PLN"), "Planned")
        .otherwise(lit("UNKNOWN"))
    )
)

# 3. Define Quality Rules
valid_condition = (
    col("event_date").isNotNull()
    & col("latitude").isNotNull()
    & col("longitude").isNotNull()
    & (col("normalized_status") != "UNKNOWN")
)

# 4. Split into Valid vs Quarantined
df_passed_dq = df_cleaned.filter(valid_condition)
df_dq_quarantine = df_cleaned.filter(~valid_condition) \
    .withColumn(
        "failure_reason",
        lit("invalid survey date, missing coordinates, or unrecognized status code")
    )

# 5. Handle duplicates among valid records (keep latest occurrence by site reference)
window_spec = Window.partitionBy("siteid").orderBy(col("event_timestamp").desc())
df_with_rn = df_passed_dq.withColumn("row_num", row_number().over(window_spec))

df_valid = df_with_rn.filter(col("row_num") == 1).drop("row_num")
df_dup_quarantine = df_with_rn.filter(col("row_num") > 1) \
    .drop("row_num") \
    .withColumn("failure_reason", lit("duplicate siteid"))

# 6. Combine all quarantine records and structure final output
df_all_quarantine = df_dq_quarantine.unionByName(df_dup_quarantine)
df_quarantine_final = df_all_quarantine \
    .withColumn("source_vendor", lit(vendor_name)) \
    .withColumn("quarantine_timestamp", current_timestamp()) \
    .withColumn("raw_record", to_json(struct([col(c) for c in df_cleaned.columns if c != "event_date"]))) \
    .select("raw_record", "failure_reason", "source_vendor", "quarantine_timestamp")

# 7. Refresh Silver Conformed and append quarantine rows
df_valid.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("inlap.silver.nv5_conformed")

df_quarantine_final.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("inlap.silver.quarantine_records")

# 8. Log Audit Metrics
rows_read = df_bronze.count()
rows_passed = df_valid.count()
rows_quarantined = df_quarantine_final.count()

spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES (
        '{vendor_name}', 
        '{layer_name}', 
        current_timestamp(), 
        'SUCCESS', 
        {rows_read}, 
        {rows_passed}, 
        {rows_quarantined}
    )
""")

print(f"Conformance complete for {vendor_name}. Read: {rows_read} | Passed: {rows_passed} | Quarantined: {rows_quarantined}")